In [0]:
spark.sql("USE CATALOG pyspark_dbt")

DataFrame[]

In [0]:
spark.sql("USE SCHEMA silver")

DataFrame[]

In [0]:
tables = spark.catalog.listTables()
tables_data = [(t.name, t.catalog, t.namespace, t.tableType, t.isTemporary) for t in tables]
display(spark.createDataFrame(tables_data, ["name", "catalog", "namespace", "tableType", "isTemporary"]))

name,catalog,namespace,tableType,isTemporary
customers,pyspark_dbt,List(silver),MANAGED,false
drivers,pyspark_dbt,List(silver),MANAGED,false
locations,pyspark_dbt,List(silver),MANAGED,false
payments,pyspark_dbt,List(silver),MANAGED,false
vehicles,pyspark_dbt,List(silver),MANAGED,false


In [0]:
tables = spark.catalog.listTables()
tables


[Table(name='customers', catalog='pyspark_dbt', namespace=['silver'], description=None, tableType='MANAGED', isTemporary=False),
 Table(name='drivers', catalog='pyspark_dbt', namespace=['silver'], description=None, tableType='MANAGED', isTemporary=False),
 Table(name='locations', catalog='pyspark_dbt', namespace=['silver'], description=None, tableType='MANAGED', isTemporary=False),
 Table(name='payments', catalog='pyspark_dbt', namespace=['silver'], description=None, tableType='MANAGED', isTemporary=False),
 Table(name='vehicles', catalog='pyspark_dbt', namespace=['silver'], description=None, tableType='MANAGED', isTemporary=False)]

In [0]:
customers = spark.table("pyspark_dbt.silver.customers")

In [0]:
customers = spark.table("pyspark_dbt.silver.customers")
drivers = spark.table("pyspark_dbt.silver.drivers")
locations = spark.table("pyspark_dbt.silver.locations")
payments = spark.table("pyspark_dbt.silver.payments")
vehicles = spark.table("pyspark_dbt.silver.vehicles")

In [0]:
customers.show(5)
payments.show(5)

+-----------+--------------------+----------------+-----------------+-----------+----------------------+------------------+------------+-----------+--------------------+
|customer_id|               email|    phone_number|             city|signup_date|last_updated_timestamp|            domain|   full_name|dedupCounts|   process_timestamp|
+-----------+--------------------+----------------+-----------------+-----------+----------------------+------------------+------------+-----------+--------------------+
|          1|azimmerman@ramire...|      8117241080|      Tiffanyview| 2023-12-25|   2025-09-15 15:21:05|ramirez-nelson.com| Daniel Reed|          1|2026-04-06 17:57:...|
|         10|   vritter@yahoo.com|0019670618100482|       Port Jesse| 2023-03-12|   2025-09-07 23:05:32|         yahoo.com|  Megan Dean|          1|2026-04-06 17:57:...|
|        100|newmanmelanie@fly...|   0010575302069|North Charlestown| 2022-06-22|   2025-09-14 22:47:28|    flynn-ross.org| Gary Barnes|          1|20

In [0]:
payments.printSchema()

root
 |-- payment_id: integer (nullable = true)
 |-- trip_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- payment_status: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- transaction_time: timestamp (nullable = true)
 |-- last_updated_timestamp: timestamp (nullable = true)
 |-- online_payment_status: string (nullable = true)
 |-- process_timestamp: timestamp (nullable = true)



In [0]:
payments_df = spark.table("pyspark_dbt.silver.payments")
display(payments_df)

payment_id,trip_id,customer_id,payment_method,payment_status,amount,transaction_time,last_updated_timestamp,online_payment_status,process_timestamp
1,274,126,Cash,Success,38.15,2025-09-17T13:00:12.000Z,2025-08-30T13:40:53.000Z,offline,2026-04-06T18:39:39.022Z
10,133,70,Cash,Failed,69.99,2025-07-31T13:00:12.000Z,2025-09-11T10:30:16.000Z,offline,2026-04-06T18:39:39.022Z
100,265,92,Wallet,Success,90.93,2025-07-15T13:00:12.000Z,2025-09-01T15:06:16.000Z,offline,2026-04-06T18:39:39.022Z
1000,903,43,Card,Pending,96.61,2025-08-29T13:00:12.000Z,2025-08-30T01:59:42.000Z,online_Pending,2026-04-06T18:39:39.022Z
101,761,107,Cash,Success,8.75,2025-08-10T13:00:12.000Z,2025-09-01T13:16:51.000Z,offline,2026-04-06T18:39:39.022Z
102,139,134,Wallet,Pending,59.42,2025-07-04T13:00:12.000Z,2025-08-25T19:45:46.000Z,offline,2026-04-06T18:39:39.022Z
103,604,80,Card,Success,45.54,2025-09-17T13:00:12.000Z,2025-09-13T02:33:29.000Z,online_Success,2026-04-06T18:39:39.022Z
104,434,122,Cash,Failed,73.37,2025-07-04T13:00:12.000Z,2025-09-04T19:07:43.000Z,offline,2026-04-06T18:39:39.022Z
105,168,80,Cash,Pending,68.93,2025-08-30T13:00:12.000Z,2025-08-30T18:12:56.000Z,offline,2026-04-06T18:39:39.022Z
106,814,87,Card,Failed,29.59,2025-08-02T13:00:12.000Z,2025-09-19T08:05:12.000Z,online_Failed,2026-04-06T18:39:39.022Z


In [0]:
%sql
CREATE DATABASE IF NOT EXISTS pyspark_dbt.gold;

In [0]:
from pyspark.sql.functions import *

fact_payments = payments_df.select(
    "payment_id",
    "trip_id",
    "customer_id",
    "payment_method",
    "payment_status",
    "online_payment_status",
    "amount",
    "transaction_time",
    "process_timestamp",
    "last_updated_timestamp"
)

In [0]:
fact_payments = fact_payments.withColumn(
    "processing_delay_sec",
    (unix_timestamp("process_timestamp") - unix_timestamp("transaction_time"))
)

In [0]:
fact_payments = fact_payments.withColumn(
    "is_success",
    when(col("payment_status") == "SUCCESS", 1).otherwise(0)
)

In [0]:
daily_revenue = fact_payments.groupBy(
    to_date("transaction_time").alias("payment_date")
).agg(
    sum("amount").alias("total_revenue"),
    count("*").alias("total_transactions"),
    avg("amount").alias("avg_transaction_value")
)

In [0]:
payment_method_kpi = fact_payments.groupBy("payment_method").agg(
    count("*").alias("total_transactions"),
    sum("amount").alias("total_revenue"),
    avg("amount").alias("avg_amount"),
    sum("is_success").alias("successful_payments")
)

In [0]:
customer_revenue = fact_payments.groupBy("customer_id").agg(
    sum("amount").alias("total_spent"),
    count("*").alias("total_payments"),
    avg("amount").alias("avg_payment")
)

In [0]:
failure_analysis = fact_payments.groupBy("payment_status").count()

In [0]:
fact_payments.write.mode("overwrite").saveAsTable("pyspark_dbt.gold.fact_payments")

In [0]:
daily_revenue.write.mode("overwrite").saveAsTable("pyspark_dbt.gold.daily_revenue")
payment_method_kpi.write.mode("overwrite").saveAsTable("pyspark_dbt.gold.payment_method_kpi")
customer_revenue.write.mode("overwrite").saveAsTable("pyspark_dbt.gold.customer_revenue")
failure_analysis.write.mode("overwrite").saveAsTable("pyspark_dbt.gold.payment_failure_analysis")

In [0]:
spark.table("pyspark_dbt.gold.daily_revenue").show()

+------------+------------------+------------------+---------------------+
|payment_date|     total_revenue|total_transactions|avg_transaction_value|
+------------+------------------+------------------+---------------------+
|  2025-09-11|1025.3799999999999|                18|    56.96555555555555|
|  2025-07-24| 696.4300000000001|                13|   53.571538461538466|
|  2025-09-20|            400.39|                 9|    44.48777777777778|
|  2025-07-20| 746.4600000000002|                12|    62.20500000000001|
|  2025-08-06| 971.6999999999999|                22|   44.168181818181814|
|  2025-08-22|            779.25|                12|              64.9375|
|  2025-08-16|            514.61|                 9|    57.17888888888889|
|  2025-09-17|419.78000000000003|                11|   38.161818181818184|
|  2025-06-28|            533.31|                10|   53.330999999999996|
|  2025-09-07|357.07000000000005|                10|    35.70700000000001|
|  2025-08-25|           

In [0]:
spark.sql("OPTIMIZE pyspark_dbt.gold.fact_payments")

DataFrame[path: string, metrics: struct<numFilesAdded:bigint,numFilesRemoved:bigint,filesAdded:struct<min:bigint,max:bigint,avg:double,totalFiles:bigint,totalSize:bigint>,filesRemoved:struct<min:bigint,max:bigint,avg:double,totalFiles:bigint,totalSize:bigint>,partitionsOptimized:bigint,zOrderStats:struct<strategyName:string,inputCubeFiles:struct<num:bigint,size:bigint>,inputOtherFiles:struct<num:bigint,size:bigint>,inputNumCubes:bigint,mergedFiles:struct<num:bigint,size:bigint>,numOutputCubes:bigint,mergedNumCubes:bigint>,clusteringStats:struct<inputZCubeFiles:struct<numFiles:bigint,size:bigint>,inputOtherFiles:struct<numFiles:bigint,size:bigint>,inputNumZCubes:bigint,mergedFiles:struct<numFiles:bigint,size:bigint>,numOutputZCubes:bigint>,numBins:bigint,numBatches:bigint,totalConsideredFiles:bigint,totalFilesSkipped:bigint,preserveInsertionOrder:boolean,numFilesSkippedToReduceWriteAmplification:bigint,numBytesSkippedToReduceWriteAmplification:bigint,startTimeMs:bigint,endTimeMs:bigint,

In [0]:
spark.sql("OPTIMIZE pyspark_dbt.gold.fact_payments ZORDER BY (customer_id, transaction_time)")

DataFrame[path: string, metrics: struct<numFilesAdded:bigint,numFilesRemoved:bigint,filesAdded:struct<min:bigint,max:bigint,avg:double,totalFiles:bigint,totalSize:bigint>,filesRemoved:struct<min:bigint,max:bigint,avg:double,totalFiles:bigint,totalSize:bigint>,partitionsOptimized:bigint,zOrderStats:struct<strategyName:string,inputCubeFiles:struct<num:bigint,size:bigint>,inputOtherFiles:struct<num:bigint,size:bigint>,inputNumCubes:bigint,mergedFiles:struct<num:bigint,size:bigint>,numOutputCubes:bigint,mergedNumCubes:bigint>,clusteringStats:struct<inputZCubeFiles:struct<numFiles:bigint,size:bigint>,inputOtherFiles:struct<numFiles:bigint,size:bigint>,inputNumZCubes:bigint,mergedFiles:struct<numFiles:bigint,size:bigint>,numOutputZCubes:bigint>,numBins:bigint,numBatches:bigint,totalConsideredFiles:bigint,totalFilesSkipped:bigint,preserveInsertionOrder:boolean,numFilesSkippedToReduceWriteAmplification:bigint,numBytesSkippedToReduceWriteAmplification:bigint,startTimeMs:bigint,endTimeMs:bigint,

##FACT TABLE

In [0]:
from pyspark.sql.functions import *

fact_payments = spark.table("pyspark_dbt.silver.payments")

fact_payments = fact_payments.withColumn(
    "is_success",
    when(col("payment_status") == "SUCCESS", 1).otherwise(0)
)

fact_payments = fact_payments.withColumn(
    "processing_delay_sec",
    unix_timestamp("process_timestamp") - unix_timestamp("transaction_time")
)

fact_payments.write.mode("overwrite").saveAsTable("pyspark_dbt.gold.fact_payments")

##BUSINESS DATA MARTS

#####Daily Revenue Mart

In [0]:
daily_revenue = fact_payments.groupBy(
    to_date("transaction_time").alias("date")
).agg(
    sum("amount").alias("total_revenue"),
    count("*").alias("transactions"),
    avg("amount").alias("avg_ticket_size")
)

spark.sql("CREATE SCHEMA IF NOT EXISTS gold")
daily_revenue.write.mode("overwrite").saveAsTable("gold.daily_revenue")

#####Payment Method Analysis

In [0]:
payment_kpi = fact_payments.groupBy("payment_method").agg(
    count("*").alias("transactions"),
    sum("amount").alias("revenue"),
    avg("amount").alias("avg_value"),
    sum("is_success").alias("successful_payments")
)

payment_kpi.write.mode("overwrite").saveAsTable("gold.payment_method_kpi")

#####Customer 360 View

In [0]:
customer_360 = fact_payments.groupBy("customer_id").agg(
    sum("amount").alias("lifetime_value"),
    count("*").alias("total_transactions"),
    avg("amount").alias("avg_transaction"),
    max("transaction_time").alias("last_payment_date")
)

customer_360.write.mode("overwrite").saveAsTable("gold.customer_360")

#####Failure Analysis

In [0]:
failure_kpi = fact_payments.groupBy("payment_status").count()

failure_kpi.write.mode("overwrite").saveAsTable("gold.payment_failures")

In [0]:
spark.table("gold.payment_failures").show()
spark.table("gold.daily_revenue").show()

+--------------+-----+
|payment_status|count|
+--------------+-----+
|       SUCCESS|  340|
|        FAILED|  319|
|       PENDING|  341|
+--------------+-----+

+----------+------------------+------------+------------------+
|      date|     total_revenue|transactions|   avg_ticket_size|
+----------+------------------+------------+------------------+
|2025-08-06| 971.6999999999998|          22| 44.16818181818181|
|2025-08-22|            779.25|          12|           64.9375|
|2025-09-20|            400.39|           9| 44.48777777777778|
|2025-07-24|            696.43|          13| 53.57153846153846|
|2025-09-11|1025.3799999999997|          18| 56.96555555555554|
|2025-07-20|            746.46|          12|62.205000000000005|
|2025-08-16|            514.61|           9| 57.17888888888889|
|2025-08-25| 725.5699999999999|          15| 48.37133333333333|
|2025-08-01|286.59999999999997|           8|35.824999999999996|
|2025-06-28|            533.31|          10|53.330999999999996|
|2025-

#Business Insights

####“UPI has highest success rate”

####“Most failures happen in CARD payments

In [0]:
spark.table("gold.daily_revenue").show()
spark.table("gold.payment_method_kpi").show()
spark.table("gold.customer_360").show()
spark.table("gold.payment_failures").show()

+----------+------------------+------------+------------------+
|      date|     total_revenue|transactions|   avg_ticket_size|
+----------+------------------+------------+------------------+
|2025-08-06| 971.6999999999998|          22| 44.16818181818181|
|2025-08-22|            779.25|          12|           64.9375|
|2025-09-20|            400.39|           9| 44.48777777777778|
|2025-07-24|            696.43|          13| 53.57153846153846|
|2025-09-11|1025.3799999999997|          18| 56.96555555555554|
|2025-07-20|            746.46|          12|62.205000000000005|
|2025-08-16|            514.61|           9| 57.17888888888889|
|2025-08-25| 725.5699999999999|          15| 48.37133333333333|
|2025-08-01|286.59999999999997|           8|35.824999999999996|
|2025-06-28|            533.31|          10|53.330999999999996|
|2025-08-23|            670.45|          14| 47.88928571428572|
|2025-09-17|419.78000000000003|          11|38.161818181818184|
|2025-09-07|            357.07|         